# ScamLens - 01 Data Exploration (Phase 1)

This notebook loads `data/processed/messages.parquet` (the single table built by
`python -m src.build_dataset`) and answers, with **real output only**:

1. How many messages do we have, and where do they come from?
2. What is the verdict balance (scam / suspicious / safe)?
3. Which tactic tags actually appear?
4. What do script styles, URL types and message lengths look like?
5. Are the splits leakage-safe (no template cluster in two splits)?

Run it top-to-bottom after `build_dataset`. Every number below is computed live.

In [ ]:
# --- setup: make the project importable no matter where jupyter was started ---
import sys, pathlib

%matplotlib inline
proj = pathlib.Path.cwd().resolve()
if not (proj / 'src').exists():
    proj = proj.parent            # notebook was opened from inside notebooks/
sys.path.insert(0, str(proj))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import data_path, TACTIC_TAGS

df = pd.read_parquet(data_path('messages_parquet'))
print(f'loaded {len(df)} rows x {len(df.columns)} columns')
df.head(3)

In [ ]:
# --- 1. where do the rows come from? (verdict x source) ---------------------
verdict_source = pd.crosstab(df['verdict'], df['source'])
display(verdict_source)

ax = verdict_source.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_title('Rows per verdict x source')
ax.set_ylabel('rows')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. verdict balance -----------------------------------------------------
counts = df['verdict'].value_counts()
display(counts)
counts.plot(kind='pie', autopct='%1.0f%%', figsize=(5, 5), title='verdict balance')
plt.ylabel('')
plt.show()

In [ ]:
# --- 3. tactic tag frequency (multi-label: one row can add several) --------
from src.data_checks import as_tactics_list

tactic_counts = {}
for tags in df['tactics']:
    for t in as_tactics_list(tags):
        tactic_counts[t] = tactic_counts.get(t, 0) + 1

tac = pd.Series(tactic_counts).reindex(TACTIC_TAGS, fill_value=0).sort_values(ascending=True)
ax = tac.plot(kind='barh', figsize=(8, 6), title='tactic tag frequency')
ax.set_xlabel('rows with this tag')
plt.tight_layout()
plt.show()
print('tags with zero rows:', [t for t in TACTIC_TAGS if tac[t] == 0])

In [ ]:
# --- 4. script_style and url_type distributions ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['script_style'].value_counts().plot(kind='bar', ax=axes[0], rot=0, title='script_style')
df['url_type'].value_counts().plot(kind='bar', ax=axes[1], rot=30, title='url_type')
plt.tight_layout()
plt.show()
display(df['script_style'].value_counts())
display(df['url_type'].value_counts())

In [ ]:
# --- 5. message length stats (scrubbed text) --------------------------------
lens = pd.DataFrame({
    'chars': df['text'].str.len(),
    'words': df['text'].str.split().str.len(),
})
display(lens.describe().round(1))
lens['chars'].plot(kind='hist', bins=30, title='message length (chars)')
plt.xlabel('characters')
plt.show()

In [ ]:
# --- 6. cluster stats and split balance (leakage sanity) ---------------------
sizes = df['template_cluster'].value_counts()
print(f'clusters: {len(sizes)} | singletons: {(sizes == 1).sum()} | largest: {sizes.max()}')

split_verdict = pd.crosstab(df['split'], df['verdict'])
display(split_verdict)

from src.data_checks import leakage_check
status, problems = leakage_check(df)
print('LEAKAGE CHECK:', status)
for p in problems:
    print('  -', p)

In [ ]:
# --- 7. read 3 real examples per verdict (scrubbed text only) ---------------
for verdict in ['scam', 'suspicious', 'safe']:
    print('=' * 70)
    print(verdict.upper())
    print('=' * 70)
    subset = df[df['verdict'] == verdict]
    if subset.empty:
        print('(no rows yet)')
    for _, row in subset.head(3).iterrows():
        print(f"  [{row['id']} | {row['source']} | {row['script_style']}] {row['text'][:160]}")
        print(f"      tactics={row['tactics']}  url_type={row['url_type']}  split={row['split']}")
    print()

## What I notice

_Written after actually running this notebook on 2026-10-20 (5,648 rows).
Update these bullets every time the dataset changes._

- **The class balance is extreme and synthetic/public-heavy**: 4,857 safe
  (86%), 750 suspicious, only 41 scam (0.7%). Of the scams, 36 are synthetic
  templates and just 5 are real labeled messages - so the real-scam count,
  not the total row count, is the true bottleneck.
- **This is not yet a Hinglish dataset**: 5,627 rows are English and only 21
  roman_hinglish, because the 5,574 downloaded public rows are English.
  Collecting consensually-sourced Hinglish messages is priority #1.
- **Every tactic tag has at least one example** (zero rows: []), but that
  coverage comes mostly from the 63 synthetic templates. Each tag needs
  real examples before Phase 2 metrics mean anything.
- **Template clustering works**: 5,058 clusters over 5,648 rows, 93%
  singletons, largest cluster 31 near-identical messages (a repeated
  English spam line from the public set). LEAKAGE CHECK: PASS.
- **Splits are honest but empty**: everything landed in train (val/test = 0)
  because synthetic/public_weak are train-only and only 11 real messages
  exist (0.2% dated). The tool warned loudly - there is currently nothing
  to evaluate on, and no evaluation number should be trusted yet.